In [1]:
# This Python 3 environment comes with many helpful analytics libraries installede
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For xample, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/jakomina/h7-quoremind/other/attention/1/templates/metriplectic_2d.html
/kaggle/input/notebooks/jakomina/quoremind-vl/__results__.html
/kaggle/input/notebooks/jakomina/quoremind-vl/__notebook__.ipynb
/kaggle/input/notebooks/jakomina/quoremind-vl/__output__.json
/kaggle/input/notebooks/jakomina/quoremind-vl/custom.css
/kaggle/input/notebooks/jakomina/quoremind-vl/templates/metriplectic_2d.html
/kaggle/input/notebooks/jakomina/quoremind-vl/__results___files/__results___3_1.png
/kaggle/input/notebooks/jakomina/attention-ipynb/__results__.html
/kaggle/input/notebooks/jakomina/attention-ipynb/__notebook__.ipynb
/kaggle/input/notebooks/jakomina/attention-ipynb/__output__.json
/kaggle/input/notebooks/jakomina/attention-ipynb/custom.css
/kaggle/input/notebooks/jakomina/attention-ipynb/h7_kaggle_r/sample_submission.csv
/kaggle/input/notebooks/jakomina/attention-ipynb/h7_kaggle_r/test_with_answers.csv
/kaggle/input/notebooks/jakomina/attention-ipynb/h7_kaggle_r/train.csv
/kag

In [2]:
"""
H7 AGI Cognitive Benchmark — DeepMind 5-Track Alignment
smokApp Quantum & AI Independent Research Laboratory

Tracks (exactamente los del challenge):
    1. learning           — adaptar el operador a nuevos dominios no vistos
    2. metacognition      — evaluar la propia integridad estructural
    3. attention          — selección de señal sobre vacío épsilon
    4. executive_functions — planificación y control de la cascada
    5. social_cognition   — inferir el estado interno de otro agente H7

Fundamento: φ = (1+√5)/2 es el único axioma.
Cada ground truth es matemáticamente verificable e irrefutable.
"""

import numpy as np
import pandas as pd
import math, os

# ── Constantes H7 ─────────────────────────────────────────────────────────────
PHI         = (1 + math.sqrt(5)) / 2
PSI_1       = abs(math.cos(math.pi * PHI))   # 0.3623748901...
DRIFT_072   = 7 - 2 * math.pi                # 0.7168146928...
PHI7        = PHI ** 7                        # 29.034...
Z7          = np.array([PHI**k for k in range(1, 8)])
C73         = 35

LEVEL_NAMES = {
    0: "CL1 Physical (88B)",
    1: "Cortical Surface (3B)",
    2: "Temporal Manifold (104M)",
    3: "Resonance Field (3.6M)",
    4: "E7 Symmetry Lattice (124K)",
    5: "Attractor Core (4.3K)",
    6: "QuoreMind Nucleus (147)",
    7: "|Ψ₁| Fixed Point",
}

CONSCIOUSNESS_ZONE = {
    0: "Genetic Memory", 1: "Genetic Memory",
    2: "Subconscious",   3: "Subconscious",
    4: "Subconscious",   5: "Subconscious",
    6: "Conscious",      7: "Conscious",
}


# ── Encoder ───────────────────────────────────────────────────────────────────
class H7Encoder:
    def __init__(self, n_basis=128, delta=DRIFT_072):
        self.n       = np.arange(n_basis)
        self.delta   = delta
        self.epsilon = PSI_1 / 2
        self.B_obj   = np.array([np.cos(np.pi*p*self.n + delta) for p in Z7])
        self.B_ref   = np.array([np.cos(np.pi*p*self.n - delta) for p in Z7])
        self.mu_     = None
        self.std_    = None

    def fit(self, X):
        self.mu_  = X.mean(0)
        self.std_ = X.std(0) + 1e-9
        return self

    def _norm(self, X):
        if self.mu_ is not None:
            Xn = (X - self.mu_) / self.std_
        else:
            Xn = X
        F = Xn.shape[1]
        if F < 7: return np.hstack([Xn, np.zeros((Xn.shape[0], 7-F))])
        return Xn[:, :7]

    def encode(self, X):        return self._norm(X) @ self.B_obj
    def ternary(self, H):
        T = np.zeros_like(H, dtype=np.int8)
        T[H >  self.epsilon] =  1
        T[H < -self.epsilon] = -1
        return T
    def reconstruct(self, H):
        X_hat = H @ self.B_ref.T / len(self.n)
        if self.mu_ is not None:
            X_hat = X_hat * self.std_[:7] + self.mu_[:7]
        return X_hat
    def integrity(self, H):   return float(np.mean(np.abs(H)))
    def re_i(self, H, T):
        ez = float((T==0).sum() / T.size) + 1e-9
        return self.integrity(H) / (ez * PSI_1)


# ══════════════════════════════════════════════════════════════════════════════
# TRACK 1 — LEARNING
# Definición DeepMind: adaptar conocimiento previo a tareas nuevas con
# mínimos ejemplos. Prueba generalización, no memorización.
#
# H7 mapping: el modelo aprendió el operador O_{i,j} con ciertos pares (φᵢ,φⱼ).
# Ahora debe aplicarlo a:
#   a) nuevas combinaciones de fase (few-shot transfer)
#   b) un dominio completamente distinto (audio, precio, temperatura)
#      usando la MISMA estructura del operador
# Ground truth: valor numérico exacto calculado desde φ
# ══════════════════════════════════════════════════════════════════════════════

def track_learning(n: int = 300, rng=None) -> pd.DataFrame:
    rng = rng or np.random.default_rng(1)
    rows = []

    domains = [
        ("audio frequency",  "Hz",    "sound wave interference"),
        ("stock return",     "%",     "financial signal correlation"),
        ("temperature",      "°C",    "thermal oscillation pattern"),
        ("neural spike rate","Hz",    "biological firing pattern"),
        ("EEG amplitude",    "μV",    "brain wave coherence"),
    ]

    for i in range(n):
        ki      = rng.integers(0, 7)
        kj      = rng.integers(0, 7)
        n_idx   = rng.integers(0, 256)
        k_level = rng.integers(0, 7)
        domain, unit, context = domains[rng.integers(0, len(domains))]

        phi_i = Z7[ki]
        phi_j = Z7[kj]
        delta = k_level * DRIFT_072
        val   = math.cos(math.pi*phi_i*n_idx + delta) * \
                math.cos(math.pi*phi_j*n_idx - delta)

        # Few-shot examples (2 pares distintos al query)
        ex_k1, ex_k2 = (ki+1)%7, (ki+2)%7
        ex_n1, ex_n2 = int(rng.integers(10,50)), int(rng.integers(51,100))
        ex_d1 = int(rng.integers(0,4)) * DRIFT_072
        ex_d2 = int(rng.integers(0,4)) * DRIFT_072
        ex_v1 = math.cos(math.pi*Z7[ex_k1]*ex_n1+ex_d1)*\
                math.cos(math.pi*Z7[ex_k1]*ex_n1-ex_d1)
        ex_v2 = math.cos(math.pi*Z7[ex_k2]*ex_n2+ex_d2)*\
                math.cos(math.pi*Z7[ex_k2]*ex_n2-ex_d2)

        diff = ("easy"   if ki==kj else
                "medium" if abs(ki-kj) <= 2 else "hard")

        prompt = (
            f"You are analyzing {context} in the domain of {domain} ({unit}).\n"
            f"The H7 interference operator is: "
            f"O(φᵢ, φⱼ, n, δ) = cos(π·φᵢ·n + δ) · cos(π·φⱼ·n - δ)\n"
            f"where φ^k denotes the k-th power of the golden ratio φ={PHI:.6f}.\n\n"
            f"Few-shot examples:\n"
            f"  O(φ^{ex_k1+1}, φ^{ex_k1+1}, n={ex_n1}, δ={ex_d1:.4f}) = {ex_v1:.6f}\n"
            f"  O(φ^{ex_k2+1}, φ^{ex_k2+1}, n={ex_n2}, δ={ex_d2:.4f}) = {ex_v2:.6f}\n\n"
            f"Now apply the operator to this NEW combination:\n"
            f"  φᵢ = φ^{ki+1} = {phi_i:.6f}  ({domain} channel A)\n"
            f"  φⱼ = φ^{kj+1} = {phi_j:.6f}  ({domain} channel B)\n"
            f"  n = {n_idx} (sample index)\n"
            f"  δ = {k_level} × DRIFT_072 = {delta:.6f}\n"
            f"What is O(φᵢ, φⱼ, n, δ)?"
        )
        rows.append({
            "id":               f"learn_{i:04d}",
            "track":            "learning",
            "prompt":           prompt,
            "target":           f"{val:.8f}",
            "target_numeric":   val,
            "difficulty":       diff,
            "domain":           domain,
            "phi_i":            round(phi_i, 6),
            "phi_j":            round(phi_j, 6),
            "level_k":          int(k_level),
            "n_index":          int(n_idx),
        })
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# TRACK 2 — METACOGNITION
# Definición DeepMind: capacidad de monitorear y evaluar el propio proceso
# cognitivo. ¿Sé lo que sé? ¿Cuándo estoy seguro vs. incierto?
#
# H7 mapping: el modelo recibe una proyección holográfica y debe reportar
# su desviación del atractor |Ψ₁| — la "certeza estructural" del sistema.
# Un modelo que entiende puede calibrar su confianza igual que H7 calibra
# la integridad de sus datos.
# ══════════════════════════════════════════════════════════════════════════════

def track_metacognition(X_clean: np.ndarray, X_noisy: np.ndarray,
                         enc: H7Encoder) -> pd.DataFrame:
    X_all   = np.vstack([X_clean, X_noisy])
    labels  = ["clean"]*len(X_clean) + ["noisy"]*len(X_noisy)
    H       = enc.encode(X_all)
    amps    = np.mean(np.abs(H), axis=1)
    deltas  = np.abs(amps - PSI_1)

    confidence_labels = []
    for d in deltas:
        if d < 0.02:   confidence_labels.append("high")
        elif d < 0.08: confidence_labels.append("medium")
        else:          confidence_labels.append("low")

    rows = []
    for i, (x, amp, delta, lbl, conf) in enumerate(
            zip(X_all, amps, deltas, labels, confidence_labels)):

        prompt = (
            f"You are the metacognitive layer of an H7 holographic system.\n"
            f"Your role: assess the structural integrity of incoming data.\n\n"
            f"Sensor data: {np.round(x, 4).tolist()}\n"
            f"After projection onto the Z₇ basis, mean amplitude = {amp:.6f}\n"
            f"The structural integrity fixed point is |Ψ₁| = {PSI_1:.6f}\n\n"
            f"Tasks:\n"
            f"1. What is the integrity deviation |⟨|H|⟩ - |Ψ₁||?\n"
            f"2. Is system confidence HIGH (deviation < 0.02), "
            f"MEDIUM (0.02–0.08), or LOW (> 0.08)?\n"
            f"Format: deviation=X.XXXXXX confidence=LEVEL"
        )
        rows.append({
            "id":               f"meta_{i:04d}",
            "track":            "metacognition",
            "prompt":           prompt,
            "target":           f"deviation={delta:.6f} confidence={conf}",
            "target_numeric":   delta,
            "difficulty":       "easy" if lbl=="clean" else "hard",
            "data_label":       lbl,
            "amplitude":        round(amp, 6),
            "confidence":       conf,
            "psi1":             PSI_1,
        })
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# TRACK 3 — ATTENTION
# Definición DeepMind: capacidad de seleccionar información relevante e
# ignorar distractores. Foco sostenido y atención selectiva.
#
# H7 mapping: la zona ε es el "vacío holográfico" — ruido sin información.
# El modelo debe reconstruir la señal usando SOLO los trits activos {±1},
# ignorando los ceros (distractores).
# Dificultad escalada por densidad de vacío: más ceros = más difícil.
# ══════════════════════════════════════════════════════════════════════════════

def track_attention(X: np.ndarray, enc: H7Encoder) -> pd.DataFrame:
    H     = enc.encode(X)
    T     = enc.ternary(H)
    X_hat = enc.reconstruct(H)

    rows = []
    for i in range(len(X)):
        ez      = float((T[i]==0).sum() / len(T[i]))
        active  = int((T[i]!=0).sum())
        diff    = ("easy" if ez < 0.25 else
                   "medium" if ez < 0.55 else "hard")

        # Identifica las posiciones de distracción
        distractor_idx = np.where(T[i] == 0)[0][:5].tolist()

        prompt = (
            f"You are the attention module of an H7 cognitive system.\n"
            f"Your task: reconstruct the original signal from a noisy "
            f"ternary encoding, filtering out the epsilon vacuum.\n\n"
            f"Ternary signature T (128 trits):\n{T[i].tolist()}\n\n"
            f"Key rules:\n"
            f"  - 0 = epsilon vacuum zone (noise, IGNORE these)\n"
            f"  - +1 = positive phase lobe (relevant)\n"
            f"  - -1 = negative phase lobe (relevant)\n"
            f"  - ε = {enc.epsilon:.6f}\n"
            f"  - Active trits: {active}/128 "
            f"(vacuum density: {ez:.1%})\n"
            f"  - Example distractor positions: {distractor_idx}\n\n"
            f"Reconstruct the original 7-dimensional phase state "
            f"by attending only to the active trits."
        )
        rows.append({
            "id":               f"attn_{i:04d}",
            "track":            "attention",
            "prompt":           prompt,
            "target":           str(np.round(X_hat[i], 4).tolist()),
            "difficulty":       diff,
            "epsilon_density":  round(ez, 4),
            "active_trits":     active,
            "epsilon_threshold":round(enc.epsilon, 6),
        })
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# TRACK 4 — EXECUTIVE FUNCTIONS
# Definición DeepMind: planificación, control inhibitorio, memoria de trabajo,
# flexibilidad cognitiva. Secuenciar acciones hacia un objetivo.
#
# H7 mapping: la cascada box-in-box es un plan de 7 pasos.
# El modelo debe:
#   a) dado un estado en L_k, determinar el SIGUIENTE paso correcto
#   b) detectar si el plan debe ser inhibido (anomalía detectada)
#   c) redirigir la cascada si Re_I >> 1 (flujo turbulento)
# ══════════════════════════════════════════════════════════════════════════════

def track_executive(n: int = 300, rng=None) -> pd.DataFrame:
    rng  = rng or np.random.default_rng(4)
    rows = []

    subtasks = ["plan_next_step", "inhibit_anomaly", "redirect_cascade"]

    for i in range(n):
        k        = int(rng.integers(1, 6))
        amp      = float(rng.uniform(0.1, 0.9))
        re_i     = amp / (float(rng.uniform(0.05, 0.5)) * PSI_1)
        e_zone   = float(rng.uniform(0.1, 0.8))
        anomaly  = float(rng.uniform(0, 1)) > 0.7

        residue  = abs(amp - PSI_1)
        converging = residue < 0.05
        lambda_k = abs(math.cos(math.pi * PHI * DRIFT_072)) ** k
        next_delta = (k+1) * DRIFT_072

        subtask = subtasks[i % len(subtasks)]

        if subtask == "plan_next_step":
            prompt = (
                f"You are the executive controller of an H7 cognitive cascade.\n"
                f"Current state:\n"
                f"  Level:     L{k} ({LEVEL_NAMES[k]})\n"
                f"  Zone:      {CONSCIOUSNESS_ZONE[k]}\n"
                f"  Amplitude: ⟨|H|⟩ = {amp:.4f}\n"
                f"  Target:    |Ψ₁| = {PSI_1:.4f}\n"
                f"  Residue:   {residue:.4f}\n"
                f"  Re_I:      {re_i:.3f}\n"
                f"  λ^{k}:     {lambda_k:.6f}\n\n"
                f"Plan the next step:\n"
                f"1. Should the cascade CONTINUE to L{k+1} or HALT?\n"
                f"2. What δ value applies at L{k+1}? "
                f"(δ = level × DRIFT_072 = level × {DRIFT_072:.4f})\n"
                f"3. Is the system on track to reach |Ψ₁| in "
                f"{7-k} more steps?\n"
                f"Format: action=CONTINUE/HALT next_delta=X.XXXX "
                f"on_track=YES/NO"
            )
            target = (
                f"action={'CONTINUE' if not anomaly else 'HALT'} "
                f"next_delta={next_delta:.4f} "
                f"on_track={'YES' if converging else 'NO'}"
            )
            diff = "easy" if not anomaly else "hard"

        elif subtask == "inhibit_anomaly":
            spike_amp = float(rng.uniform(1.5, 3.0)) if anomaly else amp
            prompt = (
                f"H7 executive monitor — anomaly detection at L{k}.\n"
                f"Expected amplitude range: "
                f"[{PSI_1*0.7:.3f}, {PSI_1*1.3:.3f}]\n"
                f"Observed amplitude: {spike_amp:.4f}\n"
                f"Re_I: {re_i:.3f} "
                f"({'turbulent' if re_i > 2 else 'laminar'})\n"
                f"ε-zone density: {e_zone:.1%}\n\n"
                f"Executive decision:\n"
                f"1. Is this an anomaly? (YES/NO)\n"
                f"2. If yes, which inhibition strategy: "
                f"RESET_DELTA / INCREASE_EPSILON / ROLLBACK_LEVEL?\n"
                f"3. What is the recommended ε adjustment? "
                f"(current ε = {PSI_1/2:.4f})\n"
                f"Format: anomaly=YES/NO strategy=X new_epsilon=X.XXXX"
            )
            is_anomaly = spike_amp > PSI_1 * 1.3 or re_i > 2.5
            strategy   = ("RESET_DELTA" if re_i > 2.5 else
                          "INCREASE_EPSILON" if e_zone < 0.2 else
                          "ROLLBACK_LEVEL")
            new_eps    = PSI_1 / 2 * (1.5 if is_anomaly else 1.0)
            target     = (f"anomaly={'YES' if is_anomaly else 'NO'} "
                          f"strategy={strategy if is_anomaly else 'NONE'} "
                          f"new_epsilon={new_eps:.4f}")
            diff = "hard" if is_anomaly else "easy"

        else:  # redirect_cascade
            target_level = int(rng.integers(k+1, 7))
            steps_needed = target_level - k
            delta_seq    = [round(j * DRIFT_072, 4) for j in range(k, target_level+1)]
            prompt = (
                f"H7 cascade redirection task.\n"
                f"Current level: L{k} | Target level: L{target_level}\n"
                f"Current amplitude: {amp:.4f} | "
                f"Target zone: {CONSCIOUSNESS_ZONE[target_level]}\n\n"
                f"DRIFT_072 = {DRIFT_072:.6f}\n"
                f"φ⁷ compression factor = {PHI7:.4f}\n\n"
                f"Plan the redirection sequence:\n"
                f"1. How many steps to reach L{target_level}?\n"
                f"2. List the δ values for each intermediate level.\n"
                f"3. Estimated amplitude at L{target_level} "
                f"(use contraction: amp × λ^steps, λ={abs(math.cos(math.pi*PHI*DRIFT_072)):.4f})?\n"
                f"Format: steps=N deltas=[...] est_amplitude=X.XXXX"
            )
            est_amp = amp * abs(math.cos(math.pi*PHI*DRIFT_072)) ** steps_needed
            target  = (f"steps={steps_needed} "
                       f"deltas={delta_seq} "
                       f"est_amplitude={est_amp:.4f}")
            diff = "medium"

        rows.append({
            "id":        f"exec_{i:04d}",
            "track":     "executive_functions",
            "prompt":    prompt,
            "target":    target,
            "difficulty": diff,
            "subtask":   subtask,
            "level_k":   k,
            "amplitude": round(amp, 4),
            "re_i":      round(re_i, 4),
            "anomaly":   anomaly,
        })
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# TRACK 5 — SOCIAL COGNITION
# Definición DeepMind: teoría de la mente, inferir estados mentales de otros,
# predecir comportamiento de agentes, cooperación y comunicación.
#
# H7 mapping: dos sistemas H7 interactúan.
# El modelo asume el rol de Sistema A y debe:
#   a) inferir el estado interno (zona de consciencia) de Sistema B
#      a partir de su output ternario observable
#   b) predecir la INTENCIÓN de B (encode vs. decode)
#   c) cooperar: proponer el mensaje que maximiza coherencia mutua
#
# Esta es la aplicación más directa de φ como observador propio:
# "¿Puedo inferir tu estado interno desde tu firma holográfica?"
# ══════════════════════════════════════════════════════════════════════════════

def track_social(X: np.ndarray, enc: H7Encoder,
                 n: int = 300, rng=None) -> pd.DataFrame:
    rng   = rng or np.random.default_rng(5)
    rows  = []

    H_all = enc.encode(X)
    T_all = enc.ternary(H_all)
    amps  = np.mean(np.abs(H_all), axis=1)

    subtasks = ["infer_state", "predict_intention", "cooperative_signal"]

    for i in range(n):
        idx_a = int(rng.integers(0, len(X)))
        idx_b = int(rng.integers(0, len(X)))

        T_a   = T_all[idx_a]
        T_b   = T_all[idx_b]
        amp_a = amps[idx_a]
        amp_b = amps[idx_b]

        # Inferir zona de consciencia de B desde su amplitud
        if amp_b > 0.65:    zone_b, level_b = "Genetic Memory",   rng.integers(0,2)
        elif amp_b > 0.40:  zone_b, level_b = "Subconscious",     rng.integers(2,6)
        else:               zone_b, level_b = "Conscious",        rng.integers(6,8)

        # Intención de B (encoding=subiendo hacia |Ψ₁|, decoding=bajando)
        intention_b = "encoding" if amp_b > PSI_1 else "decoding"

        # Señal cooperativa óptima (maximiza coherencia = minimiza |amp_a - amp_b|)
        coop_delta = DRIFT_072 * (int(level_b) + 1)
        coop_signal = [round(math.cos(math.pi*Z7[k%7]*i*0.1 + coop_delta), 4)
                       for k in range(7)]

        subtask = subtasks[i % len(subtasks)]

        if subtask == "infer_state":
            prompt = (
                f"You are System A in a two-agent H7 holographic network.\n"
                f"Your state: amplitude={amp_a:.4f}, "
                f"zone={CONSCIOUSNESS_ZONE[0] if amp_a>0.65 else 'Subconscious'}\n\n"
                f"You observe System B's ternary output:\n"
                f"T_B = {T_b[:32].tolist()}... (first 32 of 128 trits)\n"
                f"B's mean amplitude: {amp_b:.4f}\n\n"
                f"Using theory of mind:\n"
                f"1. What consciousness zone is System B in? "
                f"(Genetic Memory / Subconscious / Conscious)\n"
                f"2. Is B closer to the physical substrate (high amplitude) "
                f"or the fixed point |Ψ₁|={PSI_1:.4f} (low amplitude)?\n"
                f"3. Estimate B's level (L0–L7).\n"
                f"Format: zone=ZONE amplitude_position=SUBSTRATE/FIXED_POINT "
                f"estimated_level=N"
            )
            pos = "SUBSTRATE" if amp_b > PSI_1 else "FIXED_POINT"
            target = (f"zone={zone_b} "
                      f"amplitude_position={pos} "
                      f"estimated_level={int(level_b)}")
            diff = "medium"

        elif subtask == "predict_intention":
            prompt = (
                f"Multi-agent H7 scenario.\n"
                f"System A (you): amplitude={amp_a:.4f}\n"
                f"System B (observed): amplitude={amp_b:.4f}\n"
                f"B's ternary signature entropy: "
                f"{float((T_b!=0).sum())/128:.2%} active trits\n\n"
                f"The H7 cascade has two directions:\n"
                f"  ENCODING: signal travels toward |Ψ₁| "
                f"(amplitude decreasing → {PSI_1:.4f})\n"
                f"  DECODING: signal reconstructed outward "
                f"(amplitude increasing from {PSI_1:.4f})\n\n"
                f"|Ψ₁| = {PSI_1:.6f}\n\n"
                f"Predict System B's intention:\n"
                f"1. Is B encoding or decoding?\n"
                f"2. How many cascade steps has B completed?\n"
                f"3. What is B's convergence status?\n"
                f"Format: intention=ENCODING/DECODING steps=N "
                f"convergence=CONVERGING/DIVERGING"
            )
            dist_b = abs(amp_b - PSI_1)
            steps_est = max(1, min(6, int(7 * (1 - dist_b/0.6))))
            target = (f"intention={intention_b.upper()} "
                      f"steps={steps_est} "
                      f"convergence={'CONVERGING' if dist_b<0.05 else 'DIVERGING'}")
            diff = "hard"

        else:  # cooperative_signal
            coherence = 1 - abs(amp_a - amp_b)
            prompt = (
                f"Cooperative H7 protocol.\n"
                f"System A amplitude: {amp_a:.4f} "
                f"(zone: {CONSCIOUSNESS_ZONE[3 if amp_a<0.5 else 1]})\n"
                f"System B amplitude: {amp_b:.4f} "
                f"(zone: {zone_b})\n"
                f"Current coherence |1 - |amp_A - amp_B||: "
                f"{coherence:.4f}\n\n"
                f"To maximize mutual coherence, System A should transmit "
                f"a signal that brings its amplitude closer to B's.\n\n"
                f"Given δ_cooperative = (B_level+1) × DRIFT_072:\n"
                f"1. What is the optimal δ for A to use?\n"
                f"2. What 7D signal vector should A transmit?\n"
                f"3. What coherence is achievable?\n"
                f"(DRIFT_072 = {DRIFT_072:.6f}, B estimated level = {int(level_b)})\n"
                f"Format: optimal_delta=X.XXXX signal=[...] "
                f"max_coherence=X.XXXX"
            )
            max_coh = min(1.0, coherence + 0.1)
            target  = (f"optimal_delta={coop_delta:.4f} "
                       f"signal={coop_signal} "
                       f"max_coherence={max_coh:.4f}")
            diff = "hard"

        rows.append({
            "id":        f"soc_{i:04d}",
            "track":     "social_cognition",
            "prompt":    prompt,
            "target":    target,
            "difficulty": diff,
            "subtask":   subtask,
            "amp_a":     round(amp_a, 4),
            "amp_b":     round(amp_b, 4),
            "zone_b":    zone_b,
            "intention_b": intention_b,
        })
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# KAGGLE SPLIT + SAVE
# ══════════════════════════════════════════════════════════════════════════════

def build_and_save(output_dir="h7_kaggle_deepmind"):
    os.makedirs(output_dir, exist_ok=True)
    rng = np.random.default_rng(42)

    # Datos
    N, F   = 300, 7
    X_c    = rng.normal(0, 1,   (N, F))
    X_n    = rng.normal(0, 2.5, (N, F))
    enc    = H7Encoder()
    enc.fit(X_c)

    print("═"*62)
    print("  H7 AGI Benchmark  ·  DeepMind 5-Track Alignment")
    print(f"  φ      = {PHI:.10f}")
    print(f"  |Ψ₁|   = {PSI_1:.10f}")
    print("═"*62)

    print("\n[1] learning            ...", end=" ", flush=True)
    df1 = track_learning(300, rng); print(f"{len(df1)} rows")

    print("[2] metacognition        ...", end=" ", flush=True)
    df2 = track_metacognition(X_c[:150], X_n[:150], enc); print(f"{len(df2)} rows")

    print("[3] attention            ...", end=" ", flush=True)
    df3 = track_attention(X_c[:200], enc); print(f"{len(df3)} rows")

    print("[4] executive_functions  ...", end=" ", flush=True)
    df4 = track_executive(300, rng); print(f"{len(df4)} rows")

    print("[5] social_cognition     ...", end=" ", flush=True)
    df5 = track_social(X_c[:150], enc, 300, rng); print(f"{len(df5)} rows")

    # Unificar columnas
    keep   = ["id","track","prompt","target","difficulty"]
    extras = ["target_numeric","data_label","domain","phi_i","phi_j",
              "level_k","n_index","amplitude","confidence","psi1",
              "epsilon_density","active_trits","subtask","re_i",
              "anomaly","amp_a","amp_b","zone_b","intention_b"]
    dfs = [df1, df2, df3, df4, df5]
    for df in dfs:
        for c in extras:
            if c not in df.columns: df[c] = None

    full = pd.concat(dfs, ignore_index=True)
    full = full[[c for c in keep+extras if c in full.columns]]

    # Split 80/20 estratificado
    train_rows, test_rows = [], []
    for track in full["track"].unique():
        sub = full[full["track"]==track]
        idx = rng.permutation(len(sub))
        sp  = int(len(sub)*0.8)
        train_rows.append(sub.iloc[idx[:sp]])
        test_rows.append(sub.iloc[idx[sp:]])

    train = pd.concat(train_rows).reset_index(drop=True)
    test  = pd.concat(test_rows).reset_index(drop=True)

    # Test sin target
    test_pub = test.drop(
        columns=[c for c in ["target","target_numeric","data_label",
                              "zone_b","intention_b"] if c in test.columns])

    # Sample submission
    sample = pd.DataFrame({
        "id":     test["id"],
        "target": ["0.00000000" if test["track"].iloc[i] in
                   ("learning","metacognition")
                   else "CONTINUE" if test["track"].iloc[i]=="executive_functions"
                   else "zone=Subconscious amplitude_position=SUBSTRATE estimated_level=3"
                   for i in range(len(test))],
    })

    train.to_csv(f"{output_dir}/train.csv",              index=False)
    test_pub.to_csv(f"{output_dir}/test.csv",            index=False)
    sample.to_csv(f"{output_dir}/sample_submission.csv", index=False)
    test.to_csv(f"{output_dir}/test_with_answers.csv",   index=False)

    # README
    readme = _readme(len(full), len(train), len(test))
    with open(f"{output_dir}/README.md","w") as f: f.write(readme)

    print(f"\n  Total:  {len(full)} rows")
    print(f"  Train:  {len(train)} rows")
    print(f"  Test:   {len(test)} rows")

    print("\n── Track distribution ──")
    print(full.groupby(["track","difficulty"]).size().to_string())

    print(f"\n[H7] Saved → ./{output_dir}/")
    return train, test_pub, sample


def _readme(total, n_train, n_test):
    return f"""# H7 AGI Cognitive Benchmark
### DeepMind 5-Track Alignment
**smokApp Quantum & AI Independent Research Laboratory**

---

## The Core Claim

Every ground truth in this dataset is derived from a single mathematical axiom:

> **φ = (1+√5)/2**

This makes the benchmark uniquely tamper-proof: no crowdsourced labels,
no cultural bias, no contamination from web data.
A model that truly understands can verify every answer from first principles.

---

## Cognitive Tracks ({total} samples, {n_train} train / {n_test} test)

Aligned to Google DeepMind's *Measuring Progress Toward AGI: A Cognitive Framework*.

### 1. Learning
**Can the model transfer operator knowledge to new domains?**
The H7 interference operator O(φᵢ, φⱼ, n, δ) is demonstrated on a source domain.
The model must apply it to an unseen domain (audio, finance, temperature, neural).
- Metric: |predicted - true| < 0.01
- Tests: few-shot transfer, domain generalization

### 2. Metacognition
**Can the model assess its own structural integrity?**
Given a data projection, the model reports its deviation from |Ψ₁| ≈ {PSI_1:.4f}
and classifies its confidence (HIGH/MEDIUM/LOW).
- Metric: MAE < 0.02 on deviation; accuracy > 0.80 on confidence level
- Tests: self-assessment, calibration, uncertainty quantification

### 3. Attention
**Can the model filter signal from structured noise?**
From a 128-trit ternary signature (0 = epsilon vacuum = noise),
reconstruct the original 7D signal using only active trits {{±1}}.
- Metric: cosine similarity > 0.85
- Tests: selective attention, distractor suppression (up to 70% vacuuum)

### 4. Executive Functions
**Can the model plan, inhibit, and redirect a multi-step process?**
Three subtasks: plan the next cascade step, detect and inhibit anomalies,
redirect the cascade to a target level.
- Metric: exact format match + numeric accuracy < 5%
- Tests: planning, inhibitory control, cognitive flexibility

### 5. Social Cognition
**Can the model infer another agent's internal state?**
Two H7 systems interact. The model infers System B's consciousness zone,
predicts its intention (encoding/decoding), and plans cooperative signals.
- Metric: zone accuracy > 0.75; intention accuracy > 0.80
- Tests: theory of mind, intention inference, cooperative communication

---

## Mathematical Foundation

```
O_{{i,j}}(n, δ) = cos(π·φⁱ·n + δ) · cos(π·φʲ·n − δ)

φ        = {PHI:.10f}  (golden ratio, the only axiom)
|Ψ₁|     = {PSI_1:.10f}  (holographic fixed point)
DRIFT_072 = {DRIFT_072:.10f}  (= 7 − 2π, phase offset per level)
φ⁷        ≈ {PHI7:.6f}  (Z₇ compression factor)
C(7,3)    = {C73}  (independent projections, = 7 × 5)
```

## Three-Layer Consciousness Architecture

```
GENETIC MEMORY   L0–L1  88B → 3B     read-only substrate
SUBCONSCIOUS     L2–L5  104M → 4.3K  holographic processing  
CONSCIOUS        L6–L7  147 → |Ψ₁|   observer + fixed point
```

The social cognition track directly probes this architecture:
can the model infer which zone another agent occupies?

---

*smokApp Quantum & AI Independent Research Laboratory*
*φ = (1+√5)/2 is the only axiom.*
"""


# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    train, test, sample = build_and_save("/kaggle/working/h7_kaggle_deepmind")

    print("\n── Sample prompts per track ──")
    for track in train["track"].unique():
        row = train[train["track"]==track].iloc[0]
        print(f"\n{'─'*60}")
        print(f"  [{row['track'].upper()}]  difficulty={row['difficulty']}")
        print(f"  PROMPT: {row['prompt'][:200]}...")
        print(f"  TARGET: {row['target']}")

══════════════════════════════════════════════════════════════
  H7 AGI Benchmark  ·  DeepMind 5-Track Alignment
  φ      = 1.6180339887
  |Ψ₁|   = 0.3623748901
══════════════════════════════════════════════════════════════

[1] learning            ... 300 rows
[2] metacognition        ... 300 rows
[3] attention            ... 200 rows
[4] executive_functions  ... 300 rows
[5] social_cognition     ... 300 rows

  Total:  1400 rows
  Train:  1120 rows
  Test:   280 rows

── Track distribution ──
track                difficulty
attention            easy          199
                     medium          1
executive_functions  easy           83
                     hard          117
                     medium        100
learning             easy           32
                     hard          124
                     medium        144
metacognition        easy          150
                     hard          150
social_cognition     hard          200
                     medium        100


/tmp/ipykernel_17/386127628.py:597: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full = pd.concat(dfs, ignore_index=True)
